# UdaPlay — Part 1: RAG Pipeline

**Project:** UdaPlay – AI Research Agent for the Video Game Industry  
**Notebook:** `Udaplay_01_solution_project.ipynb`

This notebook:
1. Loads and processes video game JSON data files
2. Formats each game into a rich text document for embedding
3. Creates a persistent ChromaDB vector database with OpenAI embeddings
4. Demonstrates semantic search over the game dataset

## 0. Environment Setup

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("config.env")

assert os.getenv("OPENAI_API_KEY") is not None, "OPENAI_API_KEY not set in config.env"
assert os.getenv("TAVILY_API_KEY") is not None, "TAVILY_API_KEY not set in config.env"

OPENAI_BASE_URL = os.getenv(
    "OPENAI_BASE_URL",
    "https://openai.vocareum.com/v1"
)

print("✅ Environment loaded successfully")
print(f"   OpenAI Base URL : {OPENAI_BASE_URL}")

In [ ]:
import json
import glob
from pathlib import Path

import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI

print("✅ Libraries imported")

## 1. Load and Process Game JSON Files

In [ ]:
def load_game_json_files(pattern: str = "games_data_*.json") -> list[dict]:
    """
    Load and merge all game JSON files matching the pattern.
    Supports both a plain list of games and a wrapped {"games": [...]} format.
    """
    all_games = []
    files = sorted(glob.glob(pattern))
    
    if not files:
        raise FileNotFoundError(f"No files found matching pattern: {pattern}")
    
    for filepath in files:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        # Support both array and wrapped object formats
        if isinstance(data, list):
            games = data
        elif isinstance(data, dict) and "games" in data:
            games = data["games"]
        else:
            raise ValueError(f"Unrecognised format in {filepath}")
        
        all_games.extend(games)
        print(f"  Loaded {len(games):>3} games from {filepath}")
    
    return all_games


games = load_game_json_files()
print(f"\n✅ Total games loaded: {len(games)}")

In [ ]:
# Inspect one game record
print("Sample game record:")
print(json.dumps(games[0], indent=2))

## 2. Format Games into Rich Text Documents

In [ ]:
def format_game_document(game: dict) -> str:
    """
    Convert a game dict into a rich natural-language document
    optimised for semantic embedding and retrieval.
    """
    platforms = ", ".join(game.get("platforms", []))
    
    doc = f"""Game Title: {game['title']}
Developer: {game.get('developer', 'Unknown')}
Publisher: {game.get('publisher', 'Unknown')}
Release Date: {game.get('release_date', 'Unknown')}
Platforms: {platforms}
Genre: {game.get('genre', 'Unknown')}
Description: {game.get('description', '')}"""
    
    return doc


def prepare_chroma_documents(games: list[dict]) -> tuple[list[str], list[str], list[dict]]:
    """
    Prepare documents, IDs, and metadata for ChromaDB ingestion.
    
    Returns:
        documents: list of formatted text strings
        ids:       list of unique string IDs
        metadatas: list of metadata dicts (for filtering)
    """
    documents = []
    ids = []
    metadatas = []
    
    for i, game in enumerate(games):
        doc_text = format_game_document(game)
        doc_id = f"game_{i:04d}_{game['title'].replace(' ', '_').replace(':', '').lower()[:40]}"
        
        metadata = {
            "title":        game.get("title", ""),
            "developer":    game.get("developer", ""),
            "publisher":    game.get("publisher", ""),
            "release_date": game.get("release_date", ""),
            "genre":        game.get("genre", ""),
            "platforms":    ", ".join(game.get("platforms", [])),
        }
        
        documents.append(doc_text)
        ids.append(doc_id)
        metadatas.append(metadata)
    
    return documents, ids, metadatas


documents, ids, metadatas = prepare_chroma_documents(games)

print(f"✅ Prepared {len(documents)} documents")
print("\nSample document:")
print("-" * 60)
print(documents[0])
print("-" * 60)

## 3. Initialise ChromaDB with OpenAI Embeddings

In [ ]:
# Use OpenAI embeddings via ChromaDB's embedding function wrapper
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=OPENAI_BASE_URL,
    model_name="text-embedding-3-small"
)

print("✅ OpenAI embedding function configured")

In [ ]:
# Create a persistent ChromaDB client
CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "udaplay_games"

client = chromadb.PersistentClient(path=CHROMA_PATH)

# Delete existing collection if re-running to avoid duplicates
try:
    client.delete_collection(COLLECTION_NAME)
    print(f"  Deleted existing collection: {COLLECTION_NAME}")
except Exception:
    pass

# Create fresh collection with OpenAI embeddings
collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=openai_ef,
    metadata={"description": "UdaPlay video game knowledge base"}
)

print(f"✅ ChromaDB collection created: {COLLECTION_NAME}")
print(f"   Persistence path: {CHROMA_PATH}")

## 4. Add Documents to Vector Database

In [ ]:
# Add all game documents — ChromaDB calls the embedding function automatically
BATCH_SIZE = 10

for i in range(0, len(documents), BATCH_SIZE):
    batch_docs  = documents[i:i + BATCH_SIZE]
    batch_ids   = ids[i:i + BATCH_SIZE]
    batch_meta  = metadatas[i:i + BATCH_SIZE]
    
    collection.add(
        documents=batch_docs,
        ids=batch_ids,
        metadatas=batch_meta
    )
    print(f"  Added batch {i // BATCH_SIZE + 1}: documents {i+1}–{min(i+BATCH_SIZE, len(documents))}")

total = collection.count()
print(f"\n✅ Vector database populated: {total} documents stored")

## 5. Vector Store Manager Class

In [ ]:
class VectorStoreManager:
    """
    Reusable manager for the UdaPlay ChromaDB vector store.
    Provides a clean interface for semantic search used by the agent.
    """
    
    def __init__(
        self,
        chroma_path: str = "./chroma_db",
        collection_name: str = "udaplay_games",
        openai_api_key: str = None,
        openai_base_url: str = None,
    ):
        self.chroma_path = chroma_path
        self.collection_name = collection_name
        
        ef = embedding_functions.OpenAIEmbeddingFunction(
            api_key=openai_api_key or os.getenv("OPENAI_API_KEY"),
            api_base=openai_base_url or OPENAI_BASE_URL,
            model_name="text-embedding-3-small"
        )
        
        client = chromadb.PersistentClient(path=chroma_path)
        self.collection = client.get_collection(
            name=collection_name,
            embedding_function=ef
        )
    
    def search(
        self,
        query: str,
        n_results: int = 3,
        where: dict = None
    ) -> list[dict]:
        """
        Perform semantic search over the game database.
        
        Args:
            query:     Natural language query string
            n_results: Number of results to return (default 3)
            where:     Optional ChromaDB metadata filter dict
        
        Returns:
            List of result dicts with keys: document, metadata, distance
        """
        kwargs = dict(
            query_texts=[query],
            n_results=min(n_results, self.collection.count()),
            include=["documents", "metadatas", "distances"]
        )
        if where:
            kwargs["where"] = where
        
        raw = self.collection.query(**kwargs)
        
        results = []
        for doc, meta, dist in zip(
            raw["documents"][0],
            raw["metadatas"][0],
            raw["distances"][0]
        ):
            results.append({
                "document": doc,
                "metadata": meta,
                "distance": round(dist, 4),
                "similarity": round(1 - dist, 4),   # cosine similarity proxy
            })
        
        return results
    
    def count(self) -> int:
        return self.collection.count()
    
    def __repr__(self):
        return f"VectorStoreManager(collection='{self.collection_name}', docs={self.count()})"


# Instantiate the manager
vsm = VectorStoreManager()
print(vsm)

## 6. Demonstrate Semantic Search

In [ ]:
def display_search_results(query: str, results: list[dict]):
    """Pretty-print search results."""
    print(f"\n{'='*65}")
    print(f"  Query: '{query}'")
    print(f"{'='*65}")
    for i, r in enumerate(results, 1):
        meta = r["metadata"]
        print(f"\n  Result {i}  |  Similarity: {r['similarity']:.4f}")
        print(f"  Title    : {meta['title']}")
        print(f"  Developer: {meta['developer']}")
        print(f"  Released : {meta['release_date']}")
        print(f"  Platforms: {meta['platforms']}")
        print(f"  Genre    : {meta['genre']}")
    print()

In [ ]:
# Test Query 1: developer lookup
q1 = "Who developed FIFA 21?"
results1 = vsm.search(q1, n_results=2)
display_search_results(q1, results1)

In [ ]:
# Test Query 2: release date lookup
q2 = "When was God of War Ragnarok released?"
results2 = vsm.search(q2, n_results=2)
display_search_results(q2, results2)

In [ ]:
# Test Query 3: platform lookup
q3 = "What platform was Pokémon Red launched on?"
results3 = vsm.search(q3, n_results=2)
display_search_results(q3, results3)

In [ ]:
# Test Query 4: genre/concept search (semantic, not keyword)
q4 = "open world RPG with crafting and exploration"
results4 = vsm.search(q4, n_results=3)
display_search_results(q4, results4)

In [ ]:
# Test Query 5: publisher search
q5 = "games made by Rockstar Games"
results5 = vsm.search(q5, n_results=3)
display_search_results(q5, results5)

## 7. Summary

**Part 1 complete.** The RAG pipeline:

| Step | Status |
|---|---|
| Load JSON game files (array & object formats) | ✅ |
| Format into rich text documents for embedding | ✅ |
| Create persistent ChromaDB vector database | ✅ |
| Add all documents with OpenAI embeddings | ✅ |
| `VectorStoreManager` class for reusable search | ✅ |
| Demonstrate semantic search with 5 example queries | ✅ |

The `VectorStoreManager` and `CHROMA_PATH` are imported and used in **Part 2** (`Udaplay_02_solution_project.ipynb`).